# Model Experiment — `N-BEATS`

## Theory

**N-BEATS** (Neural Basis Expansion Analysis for Time Series) is a deep,
fully-connected architecture built from stacked residual blocks; each block
outputs a *backcast* (explains the input) and a *forecast* (extrapolates it).
Stacks can be **generic** (unconstrained basis, learns whatever functions help)
or **interpretable** (constrained to trend + seasonality basis functions, more
explainable but less flexible). It is trained **globally**: one model sees all
~3,000 (Store, Dept) series at once and shares weights across them, unlike
per-series ARIMA/SARIMA which fit each series independently.

**Key architectural property vs. the classical notebooks:** N-BEATS (in its
original form, as used here — `neuralforecast`'s `NBEATS`, not the extended
`NBEATSx`) is **purely univariate** — it only looks at a window of the series'
own past values (`input_size`), with **no exogenous inputs** (no Temperature,
CPI, markdowns, holiday flags) and **no static covariates** (no Store Type/Size).
That's a deliberate architectural choice worth discussing: whatever calendar/
holiday signal it picks up, it has to infer purely from the shape of the sales
history itself. The TFT notebook is the counterpoint — same competition, same
DL family, but explicitly designed to fuse static + exogenous + historical
inputs via attention.

**Direct multi-horizon forecasting:** N-BEATS predicts all `h` future steps in
one forward pass (unlike ARIMA's recursive one-step-at-a-time rollout), so `h`
must be fixed *before* training (it sizes the output layer). CV here uses a
small `h=8` model (matching the shared 8-week fold convention); the Final run
uses `h` = the full Kaggle test horizon (39 weeks) — two different model
instances, not one model reused at two horizons.

**Tracking:** Weights & Biases (the assignment explicitly allows this for
neural network architectures).

## 0. Config & environment (Kaggle vs local)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_COMPETITION = 'walmart-recruiting-store-sales-forecasting'
ON_KAGGLE = KAGGLE_INPUT.exists()

if ON_KAGGLE:
    RAW_DIR = KAGGLE_INPUT / KAGGLE_COMPETITION
    WORKING_DIR = Path('/kaggle/working')
else:
    PROJECT_ROOT = Path.cwd().parent
    RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
    WORKING_DIR = PROJECT_ROOT
    load_dotenv(PROJECT_ROOT / '.env')

def _resolve(stem):
    for name in (f'{stem}.csv', f'{stem}.csv.zip'):
        p = RAW_DIR / name
        if p.exists():
            return p
    return RAW_DIR / f'{stem}.csv'

TRAIN_CSV = _resolve('train')
TEST_CSV = _resolve('test')
FEATURES_CSV = _resolve('features')
STORES_CSV = _resolve('stores')

RANDOM_SEED = 42
TARGET = 'Weekly_Sales'
HOLIDAY_WEIGHT = 5
NON_HOLIDAY_WEIGHT = 1

if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for key in ('MLFLOW_TRACKING_URI', 'MLFLOW_TRACKING_USERNAME', 'MLFLOW_TRACKING_PASSWORD'):
            try:
                os.environ.setdefault(key, client.get_secret(key))
            except Exception:
                pass
    except Exception:
        pass

MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI')
MLFLOW_TRACKING_USERNAME = os.getenv('MLFLOW_TRACKING_USERNAME')
MLFLOW_TRACKING_PASSWORD = os.getenv('MLFLOW_TRACKING_PASSWORD')

print('On Kaggle:', ON_KAGGLE, '| raw data dir:', RAW_DIR)

## 1. Data loading helpers

In [ ]:
import pandas as pd

def _read_bool(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.upper().map({'TRUE': True, 'FALSE': False})

def load_stores():
    return pd.read_csv(STORES_CSV)

def load_features():
    df = pd.read_csv(FEATURES_CSV, parse_dates=['Date'])
    df['IsHoliday'] = _read_bool(df['IsHoliday'])
    df = df.sort_values(['Store', 'Date'])
    for col in ('CPI', 'Unemployment'):
        df[col] = df.groupby('Store')[col].transform(lambda s: s.ffill().bfill())
    return df.reset_index(drop=True)

def load_raw(split):
    path = TRAIN_CSV if split == 'train' else TEST_CSV
    df = pd.read_csv(path, parse_dates=['Date'])
    df['IsHoliday'] = _read_bool(df['IsHoliday'])
    return df

def load_merged(split='train'):
    base = load_raw(split)
    stores = load_stores()
    feats = load_features().drop(columns=['IsHoliday'])
    df = base.merge(stores, on='Store', how='left')
    df = df.merge(feats, on=['Store', 'Date'], how='left')
    df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
    return df

## 2. Metric (WMAE) & cross-validation helpers

In [ ]:
def weights_from_holiday(is_holiday):
    is_holiday = np.asarray(is_holiday).astype(bool)
    return np.where(is_holiday, HOLIDAY_WEIGHT, NON_HOLIDAY_WEIGHT).astype(float)

def wmae(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = weights_from_holiday(is_holiday)
    return float(np.sum(w * np.abs(y_true - y_pred)) / np.sum(w))

In [ ]:
def _sorted_unique_dates(dates):
    return np.sort(pd.to_datetime(dates).unique())

def time_holdout(df, weeks=8, date_col='Date'):
    uniq = _sorted_unique_dates(df[date_col])
    if weeks >= len(uniq):
        raise ValueError(f'weeks={weeks} >= number of distinct weeks {len(uniq)}')
    cutoff = uniq[-weeks]
    d = pd.to_datetime(df[date_col]).to_numpy()
    train_idx = np.where(d < cutoff)[0]
    val_idx = np.where(d >= cutoff)[0]
    return train_idx, val_idx

def expanding_splits(df, n_splits=3, horizon=8, date_col='Date'):
    uniq = _sorted_unique_dates(df[date_col])
    needed = horizon * n_splits
    if needed >= len(uniq):
        raise ValueError(f'Need > {needed} distinct weeks for {n_splits} folds of {horizon}; have {len(uniq)}.')
    d = pd.to_datetime(df[date_col]).to_numpy()
    for k in range(n_splits):
        end_offset = needed - k * horizon
        start_offset = end_offset - horizon
        val_start = uniq[-end_offset]
        val_end = uniq[-start_offset] if start_offset > 0 else None
        train_idx = np.where(d < val_start)[0]
        if val_end is None:
            val_idx = np.where(d >= val_start)[0]
        else:
            val_idx = np.where((d >= val_start) & (d < val_end))[0]
        yield train_idx, val_idx

## 3. Global-model (neuralforecast) shared helpers

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin

NF_FREQ = 'W-FRI'

def make_unique_id(df):
    return df['Store'].astype(str) + '_' + df['Dept'].astype(str)

def cold_start_fallback_table(long_train):
    """Per-unique_id recent mean, plus a global mean for series never seen in training."""
    per_id = long_train.groupby('unique_id')['y'].apply(lambda s: s.tail(8).mean())
    global_mean = float(long_train['y'].mean())
    return per_id, global_mean

def fill_cold_start(pred_df, value_col, fallback_per_id, global_mean):
    missing = pred_df[value_col].isna()
    if missing.any():
        pred_df.loc[missing, value_col] = pred_df.loc[missing, 'unique_id'].map(fallback_per_id)
        pred_df[value_col] = pred_df[value_col].fillna(global_mean)
    return pred_df

## 4. Weights & Biases setup

In [ ]:
import wandb
import numpy as np, pandas as pd

MODEL_NAME = 'NBEATS'
WANDB_PROJECT = 'walmart-sales-forecasting'
wandb.login()

## Load raw data

In [ ]:
raw_train = load_raw('train')
raw_test  = load_raw('test')
raw_train.shape, raw_test.shape

## Run 1 — Cleaning
Build the long `unique_id / ds / y` panel neuralforecast expects, reindexing
each series to a continuous weekly frequency (small gaps interpolated) — the
same gap-handling idea as the classical notebooks, just reused here for a
different data shape.

In [ ]:
def build_long_panel(df, value_col='Weekly_Sales'):
    out = pd.DataFrame({
        'unique_id': make_unique_id(df),
        'ds': pd.to_datetime(df['Date']),
        'y': df[value_col].to_numpy(dtype=float),
    })
    filled = []
    n_with_gaps = 0
    for uid, g in out.groupby('unique_id'):
        g = g.sort_values('ds').drop_duplicates('ds')
        full_idx = pd.date_range(g['ds'].min(), g['ds'].max(), freq=NF_FREQ)
        if len(full_idx) != len(g):
            n_with_gaps += 1
        s = g.set_index('ds')['y'].reindex(full_idx).interpolate(limit=4).ffill().bfill()
        filled.append(pd.DataFrame({'unique_id': uid, 'ds': full_idx, 'y': s.to_numpy()}))
    long_df = pd.concat(filled, ignore_index=True)
    return long_df, n_with_gaps

long_train, n_with_gaps = build_long_panel(raw_train)

run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='cleaning', name=f'{MODEL_NAME}_Cleaning')
wandb.log({
    'n_series': long_train['unique_id'].nunique(),
    'n_series_with_gaps': n_with_gaps,
    'n_train_rows': len(raw_train),
    'n_negative_sales': int((raw_train.Weekly_Sales < 0).sum()),
})
run.finish()
print(long_train['unique_id'].nunique(), 'series,', n_with_gaps, 'with gaps (interpolated)')

## Run 2 — "Feature Selection" (architecture/window choice)
N-BEATS has no exogenous features to select by design (see the theory note
above), so the equivalent decision is: how much history to look back
(`input_size`) and which stack composition to use (generic vs
trend+seasonality). Logged as the required `{MODEL_NAME}_Feature_Selection`
run, with the reasoning instead of a feature list.

In [ ]:
candidate_input_sizes = [26, 52]
candidate_stack_types = [
    ['identity', 'identity', 'identity'],
    ['trend', 'seasonality', 'identity'],
]
candidate_learning_rates = [1e-3, 5e-4]

run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='feature_selection', name=f'{MODEL_NAME}_Feature_Selection')
wandb.config.update({
    'candidate_input_sizes': candidate_input_sizes,
    'candidate_stack_types': candidate_stack_types,
    'candidate_learning_rates': candidate_learning_rates,
    'exogenous_features_used': 'none (architecture is purely univariate)',
})
run.finish()
print('search space:', len(candidate_input_sizes) * len(candidate_stack_types) * len(candidate_learning_rates), 'configs')

## 5. Pipeline wrapper (raw-test-ready, wraps neuralforecast)

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.losses.pytorch import MAE

class NBEATSPipeline(BaseEstimator, RegressorMixin):
    """sklearn-style wrapper so N-BEATS follows the same fit(raw_train)/predict(raw_test)
    contract as every other model in this repo. `horizon` must cover the longest
    gap (in weeks) between the end of training and the furthest requested date --
    fixed per assignment note above (direct multi-horizon architecture)."""

    def __init__(self, horizon=39, input_size=52, stack_types=None, learning_rate=1e-3, max_steps=300, random_seed=42):
        self.horizon = horizon
        self.input_size = input_size
        self.stack_types = stack_types or ['identity', 'identity', 'identity']
        self.learning_rate = learning_rate
        self.max_steps = max_steps
        self.random_seed = random_seed

    def fit(self, X, y=None):
        df = X.copy()
        df['Weekly_Sales'] = np.asarray(y, dtype=float)
        self.long_train_, _ = build_long_panel(df)
        self.fallback_per_id_, self.global_mean_ = cold_start_fallback_table(self.long_train_)

        model = NBEATS(
            h=self.horizon, input_size=self.input_size, stack_types=self.stack_types,
            learning_rate=self.learning_rate, max_steps=self.max_steps, loss=MAE(),
            random_seed=self.random_seed,
        )
        self.nf_ = NeuralForecast(models=[model], freq=NF_FREQ)
        self.nf_.fit(df=self.long_train_)
        self.model_col_ = 'NBEATS'
        return self

    def predict(self, X):
        req = pd.DataFrame({'unique_id': make_unique_id(X), 'ds': pd.to_datetime(X['Date'])})
        fcst = self.nf_.predict()
        merged = req.merge(fcst[['unique_id', 'ds', self.model_col_]], on=['unique_id', 'ds'], how='left')
        merged = fill_cold_start(merged, self.model_col_, self.fallback_per_id_, self.global_mean_)
        return merged[self.model_col_].to_numpy()

## Run 3 — Cross-validation (curated hyperparameter search)
8 configs (`input_size` x `stack_types` x `learning_rate`), each evaluated with
an 8-week expanding-window backtest (matching the shared CV convention). One
Wandb run per trial. `max_steps` is kept modest here for a reasonable search
time -- raise it for the Final run / on a machine with a GPU.

In [ ]:
import itertools

configs = [
    {'input_size': i, 'stack_types': s, 'learning_rate': lr}
    for i, s, lr in itertools.product(candidate_input_sizes, candidate_stack_types, candidate_learning_rates)
]
print(len(configs), 'configs to try')

trial_results = []
for i, cfg in enumerate(configs):
    run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='cv', name=f'{MODEL_NAME}_CV_trial{i}', config=cfg, reinit=True)
    fold_scores = []
    for k, (tr_idx, va_idx) in enumerate(expanding_splits(raw_train, n_splits=3, horizon=8)):
        tr, va = raw_train.iloc[tr_idx], raw_train.iloc[va_idx]
        pipe = NBEATSPipeline(horizon=8, max_steps=150, **cfg)
        pipe.fit(tr, tr['Weekly_Sales'])
        pred = pipe.predict(va)
        score = wmae(va['Weekly_Sales'], pred, va['IsHoliday'])
        fold_scores.append(score)
        wandb.log({'fold': k, 'fold_wmae': score})
    cv_mean = float(np.mean(fold_scores))
    wandb.log({'cv_wmae_mean': cv_mean, 'cv_wmae_std': float(np.std(fold_scores))})
    run.finish()
    trial_results.append({'config': cfg, 'cv_wmae_mean': cv_mean})
    print(f'config {i} {cfg}: CV WMAE={cv_mean:,.1f}')

best_trial = min(trial_results, key=lambda t: t['cv_wmae_mean'])
best_config = best_trial['config']
print('best config:', best_config, '| CV WMAE:', best_trial['cv_wmae_mean'])

## Run 4 — Final fit + save Pipeline
Refit the chosen config on the **full** raw_train with `horizon` = the number
of weeks in the Kaggle test period, check holdout WMAE, and save the fitted
Pipeline (`neuralforecast`'s own `nf.save(...)`, plus a pickle of the wrapper
so it can be reloaded and called exactly like the other notebooks' pipelines).
Wandb doesn't have the same "Model Registry" concept the assignment wants for
the *overall* best model -- if this architecture wins, log/register it via
MLflow's generic `pyfunc` flavor at that point (see note below).

In [ ]:
TEST_HORIZON = raw_test['Date'].nunique()
print('forecast horizon:', TEST_HORIZON, 'weeks')

run = wandb.init(project=WANDB_PROJECT, group=MODEL_NAME, job_type='final', name=f'{MODEL_NAME}_Final', config=best_config)

holdout_tr, holdout_va = time_holdout(raw_train, weeks=8)
p = NBEATSPipeline(horizon=8, max_steps=300, **best_config)
p.fit(raw_train.iloc[holdout_tr], raw_train.iloc[holdout_tr]['Weekly_Sales'])
hv = raw_train.iloc[holdout_va]
holdout_wmae = wmae(hv['Weekly_Sales'], p.predict(hv), hv['IsHoliday'])
wandb.log({'holdout_wmae': holdout_wmae})
print('holdout WMAE:', holdout_wmae)

final_pipe = NBEATSPipeline(horizon=TEST_HORIZON, max_steps=300, **best_config)
final_pipe.fit(raw_train, raw_train['Weekly_Sales'])

import pickle, pathlib
out_dir = pathlib.Path(WORKING_DIR) / 'models' / MODEL_NAME
out_dir.mkdir(parents=True, exist_ok=True)
final_pipe.nf_.save(path=str(out_dir / 'nf_model'), overwrite=True)
with open(out_dir / 'pipeline_wrapper.pkl', 'wb') as f:
    pickle.dump({'horizon': TEST_HORIZON, **best_config}, f)
wandb.log_artifact(str(out_dir), name=f'{MODEL_NAME}_pipeline', type='model')
run.finish()
print('saved pipeline to', out_dir)

> **If N-BEATS turns out to be the overall best model:** wrap `final_pipe`
> (or reload it via `NeuralForecast.load(path=...)` + the pickled wrapper
> config) in a `mlflow.pyfunc.PythonModel` and log/register that with
> `mlflow.pyfunc.log_model(...)` + `mlflow.register_model(...)` so
> `model_inference.ipynb` can still load it the same way as the sklearn-based
> Pipelines from the Model Registry.